In [1]:
# Env
NUM_AGENTS = 2
HEIGHT = 6
WIDTH = 6
SPAWN_PROB_PER_CELL = 0.05  # 0.05
DESPAWN_PROB_PER_CELL = 0.09



# Logging
NUM_ENV_CHECKPOINTS = 10


In [2]:
import sys
sys.path.append("../..")
from models.value_cnn import ValueCNNCentralized
import numpy as np
from orchard.environment import MoveAction, OrchardBasic
from tqdm import tqdm
import torch
from scipy.stats import norm
import copy
import numpy as np
import torch
import matplotlib.pyplot as plt

from models.value_cnn import Transition
from utils import ten_float
from dataclasses import dataclass
from typing import Any
from tadd_helpers.env_functions import *
from tadd_helpers.eval_functions import *
import csv
from pathlib import Path

import ast

--- PyTorch is configured to use: cuda ---


In [3]:
def update_move_state(s: State, agent_idx: int, move_vector: np.ndarray):
    """
    Simulates an agent taking an action. Does not modify in place.
    
    Returns:
        tuple: (next_state: State, reward: float)
    """
    old_agent_pos = s.agent_position(agent_idx)
    new_agent_pos = np.clip(old_agent_pos + move_vector, [0, 0], [s.H - 1, s.L - 1]) 
    
    s_new = s.copy()
    s_new.set_agent_position(agent_idx, new_agent_pos)

    reward = 0
    return s_new, reward
    

In [4]:
def update_spawn_despawn_and_pick(s: State, agent_idx: int, picked: bool):
    """Simulate the second phase. Agents do not move in this phase but apples may change."""
    reward = 0
    s_new = s.copy()
    if picked:
        picker_pos = s_new.agent_position(agent_idx)
        assert s_new.apples[tuple(picker_pos)] > 0 # must be that an agent picked up apple if picked is True
        s_new.remove_apple_at(picker_pos)
        reward = 1
    spawn_apples(s_new, SPAWN_PROB_PER_CELL)
    despawn_apples(s_new, DESPAWN_PROB_PER_CELL)
    return s_new, reward

### Get CNN Centralized Estimate Value

In [5]:
s_0 = init_empty_state(HEIGHT, WIDTH, NUM_AGENTS)
s_first = s_0.copy()
s_second = s_0.copy()
s_third = s_0.copy()
total_steps = 0
s_t = s_0.copy()
for t in tqdm(range(1000), desc="Training"):
    c = s_t.get_random_agent_id()
    assert c is not None
    
    # This is first update
    move_action = nearest_policy(s_t, c)

    s_t_plus_half, r_t = update_move_state(s_t, c, move_action.vector)
    
    # This is second update
    if s_t_plus_half.apples[s_t_plus_half.agent_position_tuple(c)] >= 1:
        pick = True
    else:
        pick = False
    s_t_plus_1, r_t_plus_half = update_spawn_despawn_and_pick(s_t_plus_half, c, pick)
    
    # next iteration prep
    s_t = s_t_plus_1
    total_steps += 1
    if t == 20:
        s_first = s_t.copy()
        s_first.name = "State after 20 steps"
    if t == 500:
        s_second = s_t.copy()
        s_second.name = "State after 500 steps"
    if t == 999:
        s_third = s_t.copy()
        s_third.name = "State after 999 steps"
states_to_save = [s_first, s_second, s_third]


Training: 100%|██████████| 1000/1000 [00:00<00:00, 17493.03it/s]


In [6]:
for state in states_to_save:
    print(state)

--- State after 20 steps (Grid: 6x6) ---

--- Agent Locations ---
  Agent 0: (1, 4)
  Agent 1: (3, 4)

--- Agents (Count) ---
. . . . . .
. . . . 1 .
. . . . . .
. . . . 1 .
. . . . . .
. . . . . .

--- Apples (Count) ---
. . . . . .
. . 1 . . .
1 1 . 1 . .
. . . 1 . .
. . . . . .
. . . . . .
--- State after 500 steps (Grid: 6x6) ---

--- Agent Locations ---
  Agent 0: (1, 4)
  Agent 1: (4, 1)

--- Agents (Count) ---
. . . . . .
. . . . 1 .
. . . . . .
. . . . . .
. 1 . . . .
. . . . . .

--- Apples (Count) ---
. 1 . . 1 .
. . . 1 . .
. . . . 1 .
. . . 1 1 .
. . . . . .
. 1 . 1 . .
--- State after 999 steps (Grid: 6x6) ---

--- Agent Locations ---
  Agent 0: (0, 4)
  Agent 1: (2, 1)

--- Agents (Count) ---
. . . . 1 .
. . . . . .
. 1 . . . .
. . . . . .
. . . . . .
. . . . . .

--- Apples (Count) ---
. . . . . .
. . . . . .
. . . . 1 .
. . . . . 1
1 . . . . 1
1 . . . . .


In [7]:
import pickle
with open("test_states.pkl", "wb") as f:
    pickle.dump(states_to_save, f)